# Chapter 30: Global Alignment

<a href="../lite/lab/index.html?path=ch30_global_alignment.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

You have four maps of different rooms, each in its own coordinate frame.
Stitch them together wrong and doors connect to walls. **Global
alignment** ensures every sub map agrees on where North is and how
everything fits together.

This chapter covers the core math: **SVD based rigid alignment**
(Procrustes) for pairwise stitching, **consistency checking** for
chains of pairwise alignments, and **joint global optimization** that
aligns all sub maps simultaneously.

```{admonition} What you will build
:class: tip

- Implement SVD based rigid alignment (Procrustes) for 2D and 3D point sets
- Align multiple sub maps into a single globally consistent map
- Verify alignment consistency using residual analysis

**Real world application:** Map stitching is used whenever multiple robots map different areas, or when a single robot builds local maps that need to be joined into a global map.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **Open3D registration** | ICP and other point cloud alignment algorithms |
| **scipy.spatial.transform** | Rotation and alignment utilities |
| **PCL (Point Cloud Library)** | C++ library with ICP, NDT, and feature based alignment |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 30.1 Map Stitching: SVD Based Rigid Alignment

Given two point sets $\{\mathbf{p}_i\}$ and $\{\mathbf{q}_i\}$ with
known correspondences, find the rotation $\mathbf{R}$ and translation
$\mathbf{t}$ minimizing:

$$\min_{\mathbf{R}, \mathbf{t}} \sum_{i=1}^{N} \| \mathbf{R} \mathbf{p}_i + \mathbf{t} - \mathbf{q}_i \|^2$$

**The Procrustes solution:**

1. Compute centroids: $\bar{\mathbf{p}} = \frac{1}{N}\sum_i \mathbf{p}_i$, $\bar{\mathbf{q}} = \frac{1}{N}\sum_i \mathbf{q}_i$
2. Center the points: $\tilde{\mathbf{p}}_i = \mathbf{p}_i - \bar{\mathbf{p}}$, $\tilde{\mathbf{q}}_i = \mathbf{q}_i - \bar{\mathbf{q}}$
3. Cross covariance: $\mathbf{H} = \sum_i \tilde{\mathbf{p}}_i \tilde{\mathbf{q}}_i^T$
4. SVD: $\mathbf{H} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$
5. Rotation: $\mathbf{R} = \mathbf{V} \text{diag}(1, \det(\mathbf{V}\mathbf{U}^T)) \mathbf{U}^T$
6. Translation: $\mathbf{t} = \bar{\mathbf{q}} - \mathbf{R} \bar{\mathbf{p}}$

In [ ]:
def svd_align(source, target):
    """
    Find R, t minimizing sum || R * source[i] + t - target[i] ||^2.
    
    Parameters:
        source: (N, 2) array of source points
        target: (N, 2) array of target points
    
    Returns:
        R: (2, 2) rotation matrix
        t: (2,) translation vector
        aligned: (N, 2) transformed source points
        rmse: root mean squared alignment error
    """
    src_mean = source.mean(axis=0)
    tgt_mean = target.mean(axis=0)
    src_c = source - src_mean
    tgt_c = target - tgt_mean
    
    H = src_c.T @ tgt_c
    U, S, Vt = np.linalg.svd(H)
    
    d = np.linalg.det(Vt.T @ U.T)
    D = np.diag([1, np.sign(d)])
    R = Vt.T @ D @ U.T
    
    t = tgt_mean - R @ src_mean
    aligned = (R @ source.T).T + t
    
    residuals = np.linalg.norm(aligned - target, axis=1)
    rmse = np.sqrt(np.mean(residuals**2))
    
    return R, t, aligned, rmse

print('SVD alignment function defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_points = 10
true_angle = np.radians(35)   # rotation from source to target
true_t = np.array([4.0, -2.0]) # translation
noise_level = 0.15             # noise on target points
# ─────────────────────────────────────────────────────────────────────────────

R_true = np.array([[np.cos(true_angle), -np.sin(true_angle)],
                    [np.sin(true_angle),  np.cos(true_angle)]])

# Source points (sub map A)
P = np.random.uniform(-3, 3, (n_points, 2))

# Target points (sub map B = rotated + translated + noise)
Q = (R_true @ P.T).T + true_t + np.random.randn(n_points, 2) * noise_level

# Align
R_est, t_est, P_aligned, rmse = svd_align(P, Q)
angle_est = np.degrees(np.arctan2(R_est[1, 0], R_est[0, 0]))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Before alignment
ax = axes[0]
ax.scatter(P[:, 0], P[:, 1], c='steelblue', s=80, marker='o', label='Source P',
           zorder=5)
ax.scatter(Q[:, 0], Q[:, 1], c='tomato', s=80, marker='^', label='Target Q',
           zorder=5)
for i in range(n_points):
    ax.plot([P[i, 0], Q[i, 0]], [P[i, 1], Q[i, 1]], 'gray', alpha=0.3, lw=1)
ax.set_title('Before alignment', fontsize=12)
ax.set_aspect('equal'); ax.legend(fontsize=9)

# After alignment
ax = axes[1]
ax.scatter(P_aligned[:, 0], P_aligned[:, 1], c='steelblue', s=80, marker='o',
           label='Aligned P', zorder=5)
ax.scatter(Q[:, 0], Q[:, 1], c='tomato', s=80, marker='^', label='Target Q',
           zorder=5)
for i in range(n_points):
    ax.plot([P_aligned[i, 0], Q[i, 0]], [P_aligned[i, 1], Q[i, 1]],
            'forestgreen', alpha=0.5, lw=1.5)
ax.set_title(f'After SVD alignment (RMSE = {rmse:.4f} m)', fontsize=12)
ax.set_aspect('equal'); ax.legend(fontsize=9)

# Per point residuals
ax = axes[2]
residuals_per_pt = np.linalg.norm(P_aligned - Q, axis=1)
ax.bar(range(n_points), residuals_per_pt, color='forestgreen', alpha=0.7)
ax.axhline(rmse, color='orange', lw=2, ls='--', label=f'RMSE = {rmse:.4f}')
ax.set_xlabel('Point index', fontsize=12)
ax.set_ylabel('Residual (m)', fontsize=12)
ax.set_title('Per point residuals after alignment', fontsize=12)
ax.legend(fontsize=10)

plt.suptitle('SVD Rigid Alignment (Procrustes)', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f'True:      angle = {np.degrees(true_angle):.1f} deg, t = [{true_t[0]:.1f}, {true_t[1]:.1f}]')
print(f'Estimated: angle = {angle_est:.1f} deg, t = [{t_est[0]:.2f}, {t_est[1]:.2f}]')
print(f'RMSE: {rmse:.4f} m (noise level = {noise_level} m)')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
noise_levels = np.linspace(0.0, 1.0, 20)
n_trials = 50
# ─────────────────────────────────────────────────────────────────────────────

# Sweep noise level and measure alignment accuracy
rmses = []
angle_errors = []
trans_errors = []

for nl in noise_levels:
    rmse_trials = []
    ae_trials = []
    te_trials = []
    for trial in range(n_trials):
        P_trial = np.random.uniform(-3, 3, (n_points, 2))
        Q_trial = (R_true @ P_trial.T).T + true_t + np.random.randn(n_points, 2) * nl
        R_t, t_t, _, rmse_t = svd_align(P_trial, Q_trial)
        a_t = np.arctan2(R_t[1, 0], R_t[0, 0])
        rmse_trials.append(rmse_t)
        ae_trials.append(np.abs(a_t - true_angle))
        te_trials.append(np.linalg.norm(t_t - true_t))
    rmses.append(np.mean(rmse_trials))
    angle_errors.append(np.degrees(np.mean(ae_trials)))
    trans_errors.append(np.mean(te_trials))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.plot(noise_levels, rmses, 'steelblue', lw=2, marker='o', ms=4)
ax.set_xlabel('Noise level (m)', fontsize=11)
ax.set_ylabel('RMSE (m)', fontsize=11)
ax.set_title('Alignment RMSE vs noise', fontsize=12)

ax = axes[1]
ax.plot(noise_levels, angle_errors, 'tomato', lw=2, marker='s', ms=4)
ax.set_xlabel('Noise level (m)', fontsize=11)
ax.set_ylabel('Rotation error (deg)', fontsize=11)
ax.set_title('Rotation error vs noise', fontsize=12)

ax = axes[2]
ax.plot(noise_levels, trans_errors, 'forestgreen', lw=2, marker='d', ms=4)
ax.set_xlabel('Noise level (m)', fontsize=11)
ax.set_ylabel('Translation error (m)', fontsize=11)
ax.set_title('Translation error vs noise', fontsize=12)

plt.suptitle('SVD alignment degrades gracefully with noise',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

print('SVD alignment is optimal in the least squares sense.')
print('Error grows linearly with noise but the solution remains unbiased.')

**Key properties of SVD alignment:**

- **Closed form:** No iteration needed. One SVD gives the answer.
- **Optimal:** Minimizes the sum of squared residuals exactly.
- **Robust to noise:** Degrades gracefully as noise increases.
- **Requires correspondences:** You must know which source point
  matches which target point. Unknown correspondences require ICP
  (iterative closest point).

## 30.2 Frame Consistency: Checking Pairwise Chains

Given 3 sub maps aligned pairwise:
- $\mathbf{T}_{AB}$: aligns B to A
- $\mathbf{T}_{BC}$: aligns C to B
- $\mathbf{T}_{AC}$: aligns C to A (direct)

The chain $\mathbf{T}_{AB} \circ \mathbf{T}_{BC}$ should equal
$\mathbf{T}_{AC}$. The **consistency error** measures how much they
disagree:

$$\epsilon = \| \mathbf{T}_{AB} \circ \mathbf{T}_{BC} - \mathbf{T}_{AC} \|$$

Large consistency errors indicate bad pairwise alignments.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(77)
n_pts_per_map = 6
n_shared = 4                  # shared points between adjacent maps
noise_align = 0.2             # noise on shared point positions
# ─────────────────────────────────────────────────────────────────────────────

def make_transform(angle_deg, tx, ty):
    a = np.radians(angle_deg)
    R = np.array([[np.cos(a), -np.sin(a)],
                  [np.sin(a),  np.cos(a)]])
    return R, np.array([tx, ty])

def apply_transform(R, t, points):
    return (R @ points.T).T + t

def compose_transforms(R1, t1, R2, t2):
    """Compose T1 then T2: T_out = T2 * T1."""
    R = R2 @ R1
    t = R2 @ t1 + t2
    return R, t

def transform_error(R1, t1, R2, t2):
    """Compute error between two transforms."""
    angle1 = np.arctan2(R1[1, 0], R1[0, 0])
    angle2 = np.arctan2(R2[1, 0], R2[0, 0])
    angle_err = np.abs(angle1 - angle2)
    trans_err = np.linalg.norm(t1 - t2)
    return angle_err, trans_err

# Three sub maps in the world frame
# True transforms from world to each sub map
R_A, t_A = make_transform(0, 0, 0)       # Map A is world frame
R_B, t_B = make_transform(20, 5, 2)      # Map B
R_C, t_C = make_transform(-15, 10, -1)   # Map C

# World frame points for each map
world_A = np.random.uniform(-3, 3, (n_pts_per_map, 2))
world_B = np.random.uniform(3, 9, (n_pts_per_map, 2))
world_C = np.random.uniform(7, 13, (n_pts_per_map, 2))

# Shared points (in world frame)
shared_AB = np.random.uniform(1, 5, (n_shared, 2))  # between A and B
shared_BC = np.random.uniform(6, 10, (n_shared, 2))  # between B and C
shared_AC = np.random.uniform(3, 8, (n_shared, 2))  # between A and C (direct)

# Points in each map's local frame
local_A = apply_transform(R_A, t_A, world_A)
local_B = apply_transform(R_B, t_B, world_B)
local_C = apply_transform(R_C, t_C, world_C)

# Shared points in each local frame (with noise)
sAB_in_A = apply_transform(R_A, t_A, shared_AB) + np.random.randn(n_shared, 2) * noise_align
sAB_in_B = apply_transform(R_B, t_B, shared_AB) + np.random.randn(n_shared, 2) * noise_align
sBC_in_B = apply_transform(R_B, t_B, shared_BC) + np.random.randn(n_shared, 2) * noise_align
sBC_in_C = apply_transform(R_C, t_C, shared_BC) + np.random.randn(n_shared, 2) * noise_align
sAC_in_A = apply_transform(R_A, t_A, shared_AC) + np.random.randn(n_shared, 2) * noise_align
sAC_in_C = apply_transform(R_C, t_C, shared_AC) + np.random.randn(n_shared, 2) * noise_align

print(f'3 sub maps, {n_shared} shared points per pair')

In [ ]:
# Compute pairwise alignments
# T_AB: aligns B to A using shared_AB points
R_AB, t_AB, _, rmse_AB = svd_align(sAB_in_B, sAB_in_A)

# T_BC: aligns C to B using shared_BC points
R_BC, t_BC, _, rmse_BC = svd_align(sBC_in_C, sBC_in_B)

# T_AC: aligns C to A using shared_AC points (direct)
R_AC_direct, t_AC_direct, _, rmse_AC = svd_align(sAC_in_C, sAC_in_A)

# Chain: align C to A via B = T_AB * T_BC
R_AC_chain, t_AC_chain = compose_transforms(R_BC, t_BC, R_AB, t_AB)

# Consistency error
angle_err, trans_err = transform_error(R_AC_chain, t_AC_chain,
                                        R_AC_direct, t_AC_direct)

print(f'Pairwise alignment RMSEs:')
print(f'  A to B: {rmse_AB:.4f} m')
print(f'  B to C: {rmse_BC:.4f} m')
print(f'  A to C (direct): {rmse_AC:.4f} m')
print(f'\nConsistency error (chain A>B>C vs direct A>C):')
print(f'  Rotation: {np.degrees(angle_err):.2f} deg')
print(f'  Translation: {trans_err:.4f} m')

In [ ]:
# Visualize: bring all maps to frame A via different paths
# Path 1: B via T_AB, C via T_AB * T_BC (chain)
B_in_A = apply_transform(R_AB, t_AB, local_B)
C_in_A_chain = apply_transform(R_AC_chain, t_AC_chain, local_C)

# Path 2: C via T_AC (direct)
C_in_A_direct = apply_transform(R_AC_direct, t_AC_direct, local_C)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(local_A[:, 0], local_A[:, 1], c='steelblue', s=80, marker='o',
           label='Map A (reference)', zorder=5)
ax.scatter(B_in_A[:, 0], B_in_A[:, 1], c='tomato', s=80, marker='^',
           label='Map B (aligned)', zorder=5)
ax.scatter(C_in_A_chain[:, 0], C_in_A_chain[:, 1], c='orange', s=80,
           marker='s', label='Map C (via B, chain)', zorder=5)
ax.scatter(C_in_A_direct[:, 0], C_in_A_direct[:, 1], c='forestgreen',
           s=80, marker='D', label='Map C (direct)', zorder=5)
# Show discrepancy
for i in range(n_pts_per_map):
    ax.plot([C_in_A_chain[i, 0], C_in_A_direct[i, 0]],
            [C_in_A_chain[i, 1], C_in_A_direct[i, 1]],
            'r-', lw=1.5, alpha=0.6)
ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.set_title('Chain vs direct: the red lines show inconsistency', fontsize=11)

ax = axes[1]
discrep = np.linalg.norm(C_in_A_chain - C_in_A_direct, axis=1)
ax.bar(range(n_pts_per_map), discrep, color='tomato', alpha=0.7)
ax.set_xlabel('Point index', fontsize=12)
ax.set_ylabel('Discrepancy (m)', fontsize=12)
ax.set_title('Per point discrepancy: chain vs direct', fontsize=12)

plt.suptitle('Frame consistency check: chain path vs direct path',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f'Mean discrepancy: {np.mean(discrep):.4f} m')
print('Ideally this should be zero. Noise causes drift in chains.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# Joint optimization to find transforms minimizing total error over ALL pairs
# ─────────────────────────────────────────────────────────────────────────────

def global_alignment_optimize(shared_pairs, initial_transforms, n_maps):
    """
    Optimize transforms T_1..T_{n-1} (map 0 is reference frame).
    
    shared_pairs: list of (map_i, map_j, pts_in_i, pts_in_j)
    initial_transforms: list of (R, t) for maps 1..n-1
    """
    # Parameterize each transform as (angle, tx, ty)
    n_params = 3 * (n_maps - 1)  # map 0 is fixed
    
    def get_transform(x, map_idx):
        if map_idx == 0:
            return np.eye(2), np.zeros(2)
        idx = 3 * (map_idx - 1)
        a = x[idx]
        R = np.array([[np.cos(a), -np.sin(a)],
                      [np.sin(a),  np.cos(a)]])
        t = x[idx+1:idx+3]
        return R, t
    
    def residuals(x):
        res = []
        for (mi, mj, pts_i, pts_j) in shared_pairs:
            Ri, ti = get_transform(x, mi)
            Rj, tj = get_transform(x, mj)
            # Transform pts from map i to world
            # T_i^{-1} maps from local_i to world
            # We need: R_i^T (pts_i - t_i) should equal R_j^T (pts_j - t_j)
            # Equivalently: R_i @ world + t_i = local_i
            # So world = R_i^T (local_i - t_i)
            Ri_inv = Ri.T
            ti_inv = -Ri.T @ ti
            Rj_inv = Rj.T
            tj_inv = -Rj.T @ tj
            
            world_i = (Ri_inv @ pts_i.T).T + ti_inv
            world_j = (Rj_inv @ pts_j.T).T + tj_inv
            
            diff = (world_i - world_j).ravel() / noise_align
            res.extend(diff)
        return np.array(res)
    
    # Initial parameter vector
    x0 = np.zeros(n_params)
    for k, (R_init, t_init) in enumerate(initial_transforms):
        idx = 3 * k
        x0[idx] = np.arctan2(R_init[1, 0], R_init[0, 0])
        x0[idx+1:idx+3] = t_init
    
    result = least_squares(residuals, x0, method='lm', max_nfev=10000)
    
    opt_transforms = []
    for k in range(n_maps - 1):
        R_opt, t_opt = get_transform(result.x, k + 1)
        opt_transforms.append((R_opt, t_opt))
    
    return opt_transforms, result.cost

print('Global alignment optimizer defined.')

In [ ]:
# Build shared pairs for joint optimization
shared_pairs = [
    (0, 1, sAB_in_A, sAB_in_B),   # A to B
    (1, 2, sBC_in_B, sBC_in_C),   # B to C
    (0, 2, sAC_in_A, sAC_in_C),   # A to C
]

# Initial transforms from pairwise alignment
# We need transforms FROM world TO local for maps B and C
# From svd_align: R_AB maps B_local to A_local
# So B_local = R_AB^{-1} (A_local - t_AB)
# Map B transform (world=A frame): R_B = R_AB^T, t_B = -R_AB^T @ t_AB

R_B_init = R_AB.T
t_B_init = -R_AB.T @ t_AB

R_C_init = R_AC_direct.T
t_C_init = -R_AC_direct.T @ t_AC_direct

initial_transforms = [(R_B_init, t_B_init), (R_C_init, t_C_init)]

opt_transforms, opt_cost = global_alignment_optimize(
    shared_pairs, initial_transforms, 3)

R_B_opt, t_B_opt = opt_transforms[0]
R_C_opt, t_C_opt = opt_transforms[1]

# Transform maps to world frame using optimized transforms
R_B_inv_opt = R_B_opt.T
t_B_inv_opt = -R_B_opt.T @ t_B_opt
R_C_inv_opt = R_C_opt.T
t_C_inv_opt = -R_C_opt.T @ t_C_opt

B_in_world_opt = (R_B_inv_opt @ local_B.T).T + t_B_inv_opt
C_in_world_opt = (R_C_inv_opt @ local_C.T).T + t_C_inv_opt

print(f'Optimization cost: {opt_cost:.6f}')

In [ ]:
# Compare pairwise sequential vs global joint alignment
# Pairwise: use chain A>B>C
B_in_A_pw = apply_transform(R_AB, t_AB, local_B)
C_in_A_pw = apply_transform(R_AC_chain, t_AC_chain, local_C)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(local_A[:, 0], local_A[:, 1], c='steelblue', s=80, marker='o',
           label='Map A', zorder=5)
ax.scatter(B_in_A_pw[:, 0], B_in_A_pw[:, 1], c='tomato', s=80, marker='^',
           label='Map B', zorder=5)
ax.scatter(C_in_A_pw[:, 0], C_in_A_pw[:, 1], c='orange', s=80, marker='s',
           label='Map C', zorder=5)
# Show ground truth
ax.scatter(world_A[:, 0], world_A[:, 1], c='forestgreen', s=30, marker='+',
           alpha=0.5)
ax.scatter(world_B[:, 0], world_B[:, 1], c='forestgreen', s=30, marker='+',
           alpha=0.5)
ax.scatter(world_C[:, 0], world_C[:, 1], c='forestgreen', s=30, marker='+',
           alpha=0.5, label='Ground truth')
ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.set_title('Sequential pairwise alignment', fontsize=12)

ax = axes[1]
ax.scatter(local_A[:, 0], local_A[:, 1], c='steelblue', s=80, marker='o',
           label='Map A', zorder=5)
ax.scatter(B_in_world_opt[:, 0], B_in_world_opt[:, 1], c='tomato', s=80,
           marker='^', label='Map B', zorder=5)
ax.scatter(C_in_world_opt[:, 0], C_in_world_opt[:, 1], c='orange', s=80,
           marker='s', label='Map C', zorder=5)
ax.scatter(world_A[:, 0], world_A[:, 1], c='forestgreen', s=30, marker='+',
           alpha=0.5)
ax.scatter(world_B[:, 0], world_B[:, 1], c='forestgreen', s=30, marker='+',
           alpha=0.5)
ax.scatter(world_C[:, 0], world_C[:, 1], c='forestgreen', s=30, marker='+',
           alpha=0.5, label='Ground truth')
ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.set_title('Global joint alignment', fontsize=12)

plt.suptitle('Sequential pairwise vs Global joint alignment',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Quantitative comparison
err_B_pw = np.mean(np.linalg.norm(B_in_A_pw - world_B, axis=1))
err_C_pw = np.mean(np.linalg.norm(C_in_A_pw - world_C, axis=1))
err_B_gl = np.mean(np.linalg.norm(B_in_world_opt - world_B, axis=1))
err_C_gl = np.mean(np.linalg.norm(C_in_world_opt - world_C, axis=1))

fig, ax = plt.subplots(figsize=(8, 5))
labels = ['Map B', 'Map C']
pw_errs = [err_B_pw, err_C_pw]
gl_errs = [err_B_gl, err_C_gl]
x_pos = np.arange(2)
w = 0.3
ax.bar(x_pos - w/2, pw_errs, w, color='tomato', alpha=0.8, label='Pairwise chain')
ax.bar(x_pos + w/2, gl_errs, w, color='steelblue', alpha=0.8, label='Global joint')
ax.set_xticks(x_pos); ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('Mean alignment error (m)', fontsize=12)
ax.set_title('Global optimization reduces alignment error', fontsize=13)
ax.legend(fontsize=11)
for i, (p, g) in enumerate(zip(pw_errs, gl_errs)):
    ax.text(i - w/2, p + 0.02, f'{p:.3f}', ha='center', fontsize=10)
    ax.text(i + w/2, g + 0.02, f'{g:.3f}', ha='center', fontsize=10)
plt.tight_layout(); plt.show()

print(f'Pairwise chain: B err = {err_B_pw:.4f}, C err = {err_C_pw:.4f}')
print(f'Global joint:   B err = {err_B_gl:.4f}, C err = {err_C_gl:.4f}')

**Observation:** Sequential pairwise alignment accumulates error along
the chain. The error in map C (aligned via A then B) is larger than
map B. Global joint optimization distributes the error across all
maps, producing a more consistent result.

---

## Capstone: Four Overlapping Sub Maps

Four sub maps arranged in a square, each overlapping with its two
neighbors:

```
   A --- B
   |     |
   D --- C
```

We implement:
1. Pairwise SVD alignment for each overlapping pair
2. Sequential alignment (A, then B via A, then C via B, then D via C)
3. Global joint optimization using ALL pairwise correspondences

The global solution should be more consistent, especially for map D
which is at the end of the chain but directly overlaps with A.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(123)
n_pts_cap = 8
n_shared_cap = 5
noise_cap = 0.2
# ─────────────────────────────────────────────────────────────────────────────

# Four sub maps in world frame
# A: top left, B: top right, C: bottom right, D: bottom left
world_maps = {
    'A': np.random.uniform(-6, -1, (n_pts_cap, 2)) + np.array([0, 5]),
    'B': np.random.uniform(1, 6, (n_pts_cap, 2)) + np.array([0, 5]),
    'C': np.random.uniform(1, 6, (n_pts_cap, 2)) + np.array([0, -1]),
    'D': np.random.uniform(-6, -1, (n_pts_cap, 2)) + np.array([0, -1]),
}

# True transforms (from world to local frame)
true_transforms = {
    'A': make_transform(0, 0, 0),
    'B': make_transform(12, 2, 0.5),
    'C': make_transform(-8, 1, -0.3),
    'D': make_transform(18, -1, 0.8),
}

# Convert world points to local frames
local_maps = {}
for name in ['A', 'B', 'C', 'D']:
    R_m, t_m = true_transforms[name]
    local_maps[name] = apply_transform(R_m, t_m, world_maps[name])

# Overlapping pairs: A-B, B-C, C-D, D-A
pairs = [('A', 'B'), ('B', 'C'), ('C', 'D'), ('D', 'A')]

# Generate shared points for each pair
shared_world = {}
shared_local = {}

for p1, p2 in pairs:
    # Place shared points between the two map regions
    c1 = world_maps[p1].mean(axis=0)
    c2 = world_maps[p2].mean(axis=0)
    mid = (c1 + c2) / 2
    shared_pts = mid + np.random.randn(n_shared_cap, 2) * 1.5
    shared_world[(p1, p2)] = shared_pts
    
    R1, t1 = true_transforms[p1]
    R2, t2 = true_transforms[p2]
    shared_local[(p1, p2)] = (
        apply_transform(R1, t1, shared_pts) + np.random.randn(n_shared_cap, 2) * noise_cap,
        apply_transform(R2, t2, shared_pts) + np.random.randn(n_shared_cap, 2) * noise_cap
    )

fig, ax = plt.subplots(figsize=(8, 8))
colors = {'A': 'steelblue', 'B': 'tomato', 'C': 'orange', 'D': 'forestgreen'}
for name in ['A', 'B', 'C', 'D']:
    pts = world_maps[name]
    ax.scatter(pts[:, 0], pts[:, 1], c=colors[name], s=80, marker='o',
              label=f'Map {name}', zorder=5)
for (p1, p2), pts in shared_world.items():
    ax.scatter(pts[:, 0], pts[:, 1], c='gray', s=30, marker='x', alpha=0.5)
    ax.plot([world_maps[p1].mean(axis=0)[0], world_maps[p2].mean(axis=0)[0]],
            [world_maps[p1].mean(axis=0)[1], world_maps[p2].mean(axis=0)[1]],
            'gray', ls=':', lw=1.5)
ax.set_aspect('equal'); ax.legend(fontsize=10)
ax.set_title('Four overlapping sub maps (world frame)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Step 1: Pairwise SVD alignment for each pair
pairwise_results = {}
for p1, p2 in pairs:
    pts_1, pts_2 = shared_local[(p1, p2)]
    R_pw, t_pw, _, rmse_pw = svd_align(pts_2, pts_1)  # align p2 to p1
    pairwise_results[(p1, p2)] = (R_pw, t_pw, rmse_pw)
    print(f'{p1} to {p2}: RMSE = {rmse_pw:.4f} m, '
          f'angle = {np.degrees(np.arctan2(R_pw[1,0], R_pw[0,0])):.1f} deg')

# Step 2: Sequential alignment (chain: A > B > C > D)
# B to A
R_BA, t_BA, _ = pairwise_results[('A', 'B')][:3]
# C to B, then to A
R_CB, t_CB, _ = pairwise_results[('B', 'C')][:3]
R_CA_seq, t_CA_seq = compose_transforms(R_CB, t_CB, R_BA, t_BA)
# D to C, then to B, then to A
R_DC, t_DC, _ = pairwise_results[('C', 'D')][:3]
R_DA_seq, t_DA_seq = compose_transforms(R_DC, t_DC, R_CA_seq, t_CA_seq)

# Transform all maps to A frame (sequential)
B_seq = apply_transform(R_BA, t_BA, local_maps['B'])
C_seq = apply_transform(R_CA_seq, t_CA_seq, local_maps['C'])
D_seq = apply_transform(R_DA_seq, t_DA_seq, local_maps['D'])

print('\nSequential alignment complete.')

In [ ]:
# Step 3: Global joint optimization
# Build shared pairs in the format expected by the optimizer
map_indices = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
global_pairs = []
for p1, p2 in pairs:
    pts_1, pts_2 = shared_local[(p1, p2)]
    global_pairs.append((map_indices[p1], map_indices[p2], pts_1, pts_2))

# Initial transforms for B, C, D (map A is reference)
# Use the inverse of the sequential alignments as initial
init_trans = [
    (R_BA.T, -R_BA.T @ t_BA),         # B
    (R_CA_seq.T, -R_CA_seq.T @ t_CA_seq),  # C
    (R_DA_seq.T, -R_DA_seq.T @ t_DA_seq),  # D
]

opt_trans, opt_cost_cap = global_alignment_optimize(
    global_pairs, init_trans, 4)

# Transform maps using global solution
global_aligned = {'A': local_maps['A'].copy()}
for k, name in enumerate(['B', 'C', 'D']):
    R_opt, t_opt = opt_trans[k]
    R_inv = R_opt.T
    t_inv = -R_opt.T @ t_opt
    global_aligned[name] = (R_inv @ local_maps[name].T).T + t_inv

print(f'Global optimization cost: {opt_cost_cap:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Sequential
ax = axes[0]
seq_maps = {'A': local_maps['A'], 'B': B_seq, 'C': C_seq, 'D': D_seq}
for name in ['A', 'B', 'C', 'D']:
    pts = seq_maps[name]
    ax.scatter(pts[:, 0], pts[:, 1], c=colors[name], s=80, marker='o',
              label=f'Map {name}', zorder=5)
    gt = world_maps[name]
    ax.scatter(gt[:, 0], gt[:, 1], c=colors[name], s=20, marker='+', alpha=0.4)
    for i in range(n_pts_cap):
        ax.plot([pts[i, 0], gt[i, 0]], [pts[i, 1], gt[i, 1]],
                color=colors[name], alpha=0.15, lw=1)
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title('Sequential pairwise alignment', fontsize=12)

# Global
ax = axes[1]
for name in ['A', 'B', 'C', 'D']:
    pts = global_aligned[name]
    ax.scatter(pts[:, 0], pts[:, 1], c=colors[name], s=80, marker='o',
              label=f'Map {name}', zorder=5)
    gt = world_maps[name]
    ax.scatter(gt[:, 0], gt[:, 1], c=colors[name], s=20, marker='+', alpha=0.4)
    for i in range(n_pts_cap):
        ax.plot([pts[i, 0], gt[i, 0]], [pts[i, 1], gt[i, 1]],
                color=colors[name], alpha=0.15, lw=1)
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title('Global joint alignment', fontsize=12)

plt.suptitle('Four sub maps: sequential vs global alignment\n'
             '(+markers = ground truth, lines = error)',
             fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
# Per-map error comparison
seq_errors = {}
global_errors = {}
for name in ['A', 'B', 'C', 'D']:
    seq_errors[name] = np.mean(np.linalg.norm(
        seq_maps[name] - world_maps[name], axis=1))
    global_errors[name] = np.mean(np.linalg.norm(
        global_aligned[name] - world_maps[name], axis=1))

fig, ax = plt.subplots(figsize=(8, 5))
map_names = ['A', 'B', 'C', 'D']
x_pos = np.arange(4)
w = 0.3
seq_vals = [seq_errors[n] for n in map_names]
gl_vals = [global_errors[n] for n in map_names]

ax.bar(x_pos - w/2, seq_vals, w, color='tomato', alpha=0.8,
       label='Sequential pairwise')
ax.bar(x_pos + w/2, gl_vals, w, color='steelblue', alpha=0.8,
       label='Global joint')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Map {n}' for n in map_names], fontsize=12)
ax.set_ylabel('Mean alignment error (m)', fontsize=12)
ax.set_title('Per map alignment error comparison', fontsize=13)
ax.legend(fontsize=11)

for i, (s, g) in enumerate(zip(seq_vals, gl_vals)):
    ax.text(i - w/2, s + 0.02, f'{s:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, g + 0.02, f'{g:.3f}', ha='center', fontsize=9)

plt.tight_layout(); plt.show()

print(f'Sequential total error: {sum(seq_vals):.4f} m')
print(f'Global total error:     {sum(gl_vals):.4f} m')
improvement = (1 - sum(gl_vals) / sum(seq_vals)) * 100
print(f'Improvement: {improvement:.1f}%')

In [ ]:
# Consistency check: D-A overlap
# In sequential alignment, D is at the end of the chain and does not
# directly use its overlap with A. In global alignment, it does.

pts_D_in_A_seq, pts_A_in_A_seq = shared_local[('D', 'A')]
D_shared_seq = apply_transform(R_DA_seq, t_DA_seq,
                                shared_local[('D', 'A')][1])
D_shared_global = (opt_trans[2][0].T @ shared_local[('D', 'A')][1].T).T + \
                  (-opt_trans[2][0].T @ opt_trans[2][1])

A_shared = shared_local[('D', 'A')][0]  # points in A frame

err_DA_seq = np.mean(np.linalg.norm(D_shared_seq - A_shared, axis=1))
err_DA_global = np.mean(np.linalg.norm(D_shared_global - A_shared, axis=1))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(['Sequential\n(D via A>B>C)', 'Global\n(D uses all links)'],
       [err_DA_seq, err_DA_global],
       color=['tomato', 'steelblue'], alpha=0.8)
ax.set_ylabel('D to A overlap error (m)', fontsize=12)
ax.set_title('Closing the loop: D to A consistency', fontsize=13)
plt.tight_layout(); plt.show()

print(f'Sequential D to A error: {err_DA_seq:.4f} m')
print(f'Global D to A error:     {err_DA_global:.4f} m')
print(f'\nThe sequential approach does not know D overlaps A.')
print('The global approach uses this link to close the loop.')

**Capstone observations:**

- **Sequential pairwise alignment** accumulates error along the chain.
  Map D, at the end of chain A > B > C > D, has the largest error.
- **Global joint alignment** uses ALL pairwise correspondences
  simultaneously. The D to A overlap acts as a loop closure, constraining
  the entire system and distributing error evenly.
- The difference between sequential and global is analogous to the
  difference between dead reckoning and SLAM: global alignment uses
  all available constraints to produce a globally consistent solution.
- In practice, global alignment is essential when stitching many
  sub maps that form loops or complex overlap patterns.

---

## Exercises

### Exercise 30.1: 3D SVD Alignment

Extend the SVD alignment to 3D. Generate 15 random 3D points, apply a
known rotation (30 degrees about the Z axis) and translation
$[1, 2, 3]^T$, add noise, then recover $\mathbf{R}$ and $\mathbf{t}$.
Verify the recovered rotation matrix has determinant +1.

In [ ]:
# Your code here

### Exercise 30.2: Minimum Points for Alignment

In 2D, the minimum number of point correspondences for a unique rigid
alignment is 2 (but 3 gives redundancy for noise). Sweep the number of
shared points from 2 to 15 and plot alignment RMSE vs. count. At what
count does adding more points stop improving accuracy?

In [ ]:
# Your code here

### Exercise 30.3: Outlier Rejection

Add 2 outlier correspondences (wrong matches) to a set of 10 correct
ones. Show that SVD alignment is corrupted. Implement a simple RANSAC
loop: randomly sample 3 correspondences, compute alignment, count
inliers (residual < threshold), keep the best. Compare RANSAC vs.
naive SVD on the corrupted data.

In [ ]:
# Your code here

### Exercise 30.4: Chain Length Scaling (challenge)

Create a chain of $N$ sub maps (each overlapping with its neighbor).
Sweep $N$ from 2 to 20. For each $N$, compare sequential pairwise
alignment vs. global joint alignment. Plot the error of the last map
vs. $N$. Show that sequential error grows linearly (or worse) while
global error stays bounded.

In [ ]:
# Your code here